In [46]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import numpy as np
import cv2

In [2]:
import os
import cv2
import numpy as np

['test_set', 'training_set']

In [3]:
train_gen = ImageDataGenerator(rescale=1./255)
test_gen  = ImageDataGenerator(rescale=1./255)

In [4]:
train_data = train_gen.flow_from_directory(
    "catsanddogs/training_set/training_set/",
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
    
)

test_data = test_gen.flow_from_directory(
    "catsanddogs/test_set/test_set/",
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

Found 8005 images belonging to 2 classes.
Found 2023 images belonging to 2 classes.


In [5]:
label_map = train_data.class_indices
inv_label_map = {v: k for k, v in label_map.items()}
print("Label mapping:", inv_label_map)

Label mapping: {0: 'cats', 1: 'dogs'}


In [7]:
model =  Sequential()
model.add(Conv2D(32, (3,3), activation='relu', input_shape = (128,128,3)))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(64, (3,3), activation='relu'))

C:\Users\memon\anaconda3\envs\vision\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        36,928 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,320 (220.00 KB)

 Trainable params: 56,320 (220.00 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.add(Flatten())
model.add(Dense(64,activation='relu'))
model.add(Dense(2,activation='softmax'))

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     3,211,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,267,778 (12.47 MB)

 Trainable params: 3,267,778 (12.47 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
history = model.fit(
    train_data,
    epochs=10,
    validation_data=test_data
)

Epoch 1/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 217s 858ms/step - accuracy: 0.6024 - loss: 0.6553 - val_accuracy: 0.6614 - val_loss: 0.6017
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 181s 720ms/step - accuracy: 0.7127 - loss: 0.5617 - val_accuracy: 0.7237 - val_loss: 0.5470
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 118s 469ms/step - accuracy: 0.7554 - loss: 0.4920 - val_accuracy: 0.7444 - val_loss: 0.5326
Epoch 4/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 123s 491ms/step - accuracy: 0.8015 - loss: 0.4244 - val_accuracy: 0.7420 - val_loss: 0.5036
Epoch 5/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 120s 476ms/step - accuracy: 0.8570 - loss: 0.3230 - val_accuracy: 0.7692 - val_loss: 0.5302
Epoch 6/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 130s 518ms/step - accuracy: 0.9187 - loss: 0.2005 - val_accuracy: 0.7756 - val_loss: 0.6505
Epoch 7/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 161s 642ms/step - accuracy: 0.9639 - loss: 0.0986 - val_accuracy: 0.7637 - val_loss: 0.9208
Epoch 8/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 169s 673ms/step - accuracy: 0.9831 -

In [18]:
model.save("cat_dog_cnn_model.h5")
print("Model Saved Successfully!")

Model Saved Successfully!


In [47]:
model = load_model("cat_dog_cnn_model.h5")
print("Model Loaded Successfully!")

Model Loaded Successfully!


In [49]:
def predict_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (128, 128))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    preds = model.predict(img)
    index = np.argmax(preds)          # no if/else
    label = inv_label_map[index]      # label from mapping

    print("\nPrediction Scores:", preds)
    print("Predicted Label:", label)

In [50]:
predict_image("dog2.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step

Prediction Scores: [[0.00567825 0.9943217 ]]
Predicted Label: dogs
